# Figure 1: Point-estimate accuracy

Compare the bootstrapped and broadband flows, six-band EAZY with its supplied R prior, and released full-COSMOS LePhare on exactly the same test IDs. LePhare sees more photometric information.

Run all cells in a Python environment with `requirements.txt` installed. The plotting style follows SpenderQ methods_paper and requires latex and dvipng. Inputs are frozen, documented snapshots in `results/figure_data/`; no training or network access is required. The final cell writes both PDF and PNG versions into `tex/figures/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'setup.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
from pzflow.plotting import (COLORS, LABELS, set_style, load_predictions,
                         load_metrics, summary, bootstrap, save_figure)
set_style()

In [ ]:
from matplotlib.colors import LogNorm
d = load_predictions(ROOT)
z = d['zspec']
names = ['cosmos_pdf_flow', 'broadband_spec_flow', 'eazy_rprior', 'lephare_full_cosmos']
titles = ['Bootstrapped flow: six bands', 'Broadband flow: six bands', 'EAZY: six bands, R prior', 'LePhare: full COSMOS']
for name in names:
    print(name, summary(z, d[f'{name}_quantiles']))

The scatter panels zoom to $0<z<5$. The annotations and manuscript table include **all** 5,083 objects. All displayed point estimates are posterior medians; the same normalized-residual definition is used for every method.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(6.6, 4.7), layout='constrained', sharex=True, sharey=True)
ax = axes.ravel()
norm = LogNorm(vmin=1, vmax=200)
for a, name, title, letter in zip(ax, names, titles, ['(a)', '(b)', '(c)', '(d)']):
    q = d[f'{name}_quantiles']
    h = a.hexbin(z, q[:, 2], gridsize=75, extent=(0, 5, 0, 5),
                 mincnt=1, norm=norm, cmap='viridis', linewidths=0, rasterized=True)
    a.plot([0, 5], [0, 5], color='#c34828', lw=.8)
    a.set(xlim=(0, 5), ylim=(0, 5), xlabel=r'$z_{\mathrm{spec}}$', title=title, aspect='equal')
    m = summary(z, q)
    text = (rf'$\sigma_{{\mathrm{{NMAD}}}}={m["sigma_nmad"]:.4f}$' + '\n' +
            rf'$\eta={100 * m["outlier_fraction"]:.2f}\%$')
    a.text(.05, .95, text, transform=a.transAxes, va='top', fontsize=7,
           bbox=dict(facecolor='white', edgecolor='none', alpha=.88, pad=2))
    a.text(.95, .05, letter, transform=a.transAxes, ha='right', color='#222222')
for a in axes[:, 0]:
    a.set_ylabel(r'Posterior median $z_{50}$')
cb = fig.colorbar(h, ax=ax, fraction=.017, pad=.015, shrink=.83)
cb.set_label('Galaxies per hexagon', fontsize=7)
save_figure(fig, ROOT, 'fig02_accuracy')
plt.show()
plt.close(fig)